# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [4]:
import sys
sys.path.append('../../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [6]:
print(docs[0])

page_content='pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025' metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': 'documents/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


In [7]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [8]:
print(document_text)

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agreem

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [9]:
from openai import OpenAI
from pydantic import BaseModel
import os
client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')

In [10]:

class TextSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [11]:
developer_prompt = """You are an AI assistant that extracts structured information from articles.Use a clearly identifiable tone of Formal Academic Writing for the summary and specify it in the Tone field.

Constraints:
- Relevance must be no longer than one paragraph.
- Summary must be concise and under 1000 tokens.
- Use the defned tone for the summary and specify it in the Tone field.
"""

In [12]:
user_prompt = f"""
Analyze the following article and extract the required structured output.

ARTICLE:
{document_text}
"""

In [13]:
response = client.responses.parse(
    model="gpt-4o-mini",
    instructions = developer_prompt,
    input = user_prompt,
    text_format=TextSummary,
)

In [14]:
from IPython.display import display, Markdown

display(Markdown(response.output_text))

{"Author":"MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article provides profound insights into the current state of Generative AI (GenAI) implementation in enterprises, highlighting a striking disparity between high adoption rates and low transformative impact across various sectors, termed the GenAI Divide. It emphasizes the challenges organizations face in scaling AI solutions, the importance of learning-capable systems, and best practices for successful implementation and procurement strategies.","Summary":"The report 'The GenAI Divide' explores the landscape of Generative AI in enterprises, revealing that despite significant investments (estimated at $30–40 billion), 95% of organizations fail to realize substantive financial impacts from their AI initiatives. The divide between successful and unsuccessful implementations—dubbed the GenAI Divide—emerges not from issues of model quality or regulatory constraints but from an organization's approach to integrating these technologies. Key findings illustrate that while generative tools like ChatGPT see widespread pilot testing (over 80% of organizations), actual deployment is considerably lower (just 5% reach production). This disparity is attributed to several factors including complex integration challenges, failure to align with operational workflows, and a fundamental learning gap in existing models. Successful organizations are characterized by their emphasis on tailored AI solutions, robust adaptation to existing workflows, and a focus on process-specific customization. In contrast, investment trends show a bias toward visible front-office improvements in sales and marketing, often overlooking back-office automation, which can yield higher returns. The report also emphasizes the importance of external partnerships for successful AI tool deployment, as organizations that utilize external vendors report significantly higher success rates compared to those relying solely on internal development. Additionally, the emergence of a 'shadow AI economy' highlights that informal usage of consumer-grade AI tools is often more impactful than formal initiatives. Overall, the document stresses that crossing the GenAI Divide requires a shift in organizational strategies, a move toward buying rather than building solutions, and a focus on tools that can learn and evolve over time.","Tone":"Formal Academic Writing","InputTokens":1602,"OutputTokens":990}

In [15]:
summarize = response.output_parsed
summarize

TextSummary(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article provides profound insights into the current state of Generative AI (GenAI) implementation in enterprises, highlighting a striking disparity between high adoption rates and low transformative impact across various sectors, termed the GenAI Divide. It emphasizes the challenges organizations face in scaling AI solutions, the importance of learning-capable systems, and best practices for successful implementation and procurement strategies.', Summary="The report 'The GenAI Divide' explores the landscape of Generative AI in enterprises, revealing that despite significant investments (estimated at $30–40 billion), 95% of organizations fail to realize substantive financial impacts from their AI initiatives. The divide between successful and unsuccessful implementations—dubbed the GenAI Divide—emerges not from issues of

In [16]:
response.model_dump()

{'id': 'resp_067ed5a07509e3a90069ebdeca6b688194bcd415444c4d07ac',
 'created_at': 1777065674.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'You are an AI assistant that extracts structured information from articles.Use a clearly identifiable tone of Formal Academic Writing for the summary and specify it in the Tone field.\n\nConstraints:\n- Relevance must be no longer than one paragraph.\n- Summary must be concise and under 1000 tokens.\n- Use the defned tone for the summary and specify it in the Tone field.\n',
 'metadata': {},
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'response',
 'output': [{'id': 'msg_067ed5a07509e3a90069ebdecb2b8881949ad9704774d4b196',
   'content': [{'annotations': [],
     'text': '{"Author":"MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article provides profound insights into the current state of Generative AI (GenAI) implementation in

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [17]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

In [18]:
summarize.Summary

"The report 'The GenAI Divide' explores the landscape of Generative AI in enterprises, revealing that despite significant investments (estimated at $30–40 billion), 95% of organizations fail to realize substantive financial impacts from their AI initiatives. The divide between successful and unsuccessful implementations—dubbed the GenAI Divide—emerges not from issues of model quality or regulatory constraints but from an organization's approach to integrating these technologies. Key findings illustrate that while generative tools like ChatGPT see widespread pilot testing (over 80% of organizations), actual deployment is considerably lower (just 5% reach production). This disparity is attributed to several factors including complex integration challenges, failure to align with operational workflows, and a fundamental learning gap in existing models. Successful organizations are characterized by their emphasis on tailored AI solutions, robust adaptation to existing workflows, and a focus

In [19]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summarize.Summary
)

In [21]:
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)


summarization_metric = SummarizationMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    assessment_questions=[
        "Does the summary capture the main idea of the original text?",
        "Are the key points mentioned from the original text?",
        "Is irrelevant information excluded from the summary?",
        "Is the summary concise without losing meaning?",
        "Does the summary remain in context to the original text without distortion?"
    ]
)

In [22]:
summarization_metric.measure(test_case)

Output()

0.47619047619047616

In [23]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {summarization_metric.score}'))
display(Markdown(f'**Reason**: {summarization_metric.reason}'))

**Score**: 0.47619047619047616

**Reason**: The score is 0.48 because the summary contains significant contradictions to the original text regarding the barriers to scaling AI and the nature of AI tools, which undermines its accuracy. Additionally, it introduces several pieces of extra information that were not present in the original text, further detracting from its fidelity. While the summary may attempt to capture the essence of the original, these issues lead to a lower score.

In [26]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

In [28]:
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate how logically structured and easy to understand the summary is.",
    evaluation_steps=[
        "Is the summary logically structured?",
        "Are sentences connected smoothly?",
        "Is the summary easy to understand?",
        "Does the summary avoid ambiguity?",
        "Is the flow of ideas consistent throughout?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [29]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summarize.Summary
)

In [32]:
result_coherence=evaluate(test_cases=[test_case], metrics=[coherence_metric])

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:000m
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:01
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:01
Evaluating 1 test case(s) in parallel 

In [ ]:
result_coherence.test_results[0].metrics_data[0].score

'No reason provided.'

In [41]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {result_coherence.test_results[0].metrics_data[0].score}'))
display(Markdown(f'**Reason**: {result_coherence.test_results[0].metrics_data[0].reason}'))

**Score**: 0.8817574482861008

**Reason**: The summary is logically structured, clearly outlining the key findings and insights from the report. Sentences are connected smoothly, making it easy to follow the argument. The language is straightforward, avoiding ambiguity, and the flow of ideas is consistent throughout, effectively highlighting the contrast between successful and unsuccessful AI implementations. However, a minor shortcoming is that while it captures the essence of the report, it could benefit from a more explicit mention of the specific barriers organizations face, which would enhance clarity on the challenges discussed.

In [44]:
# Tonality
tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the tone of the summary is appropriate and consistent.",
    evaluation_steps=[
        "Is the tone of the content in Formal Academic Writing?",
        "Is the tone consistent throughout the summary?",
        "Does the tone remain objective?",
        "Does the summary avoid biased language?",
        "Is the tone suitable for the intended audience?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [45]:
result_tone=tonality_metric.measure(test_case)

Output()

In [49]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {tonality_metric.score}'))
display(Markdown(f'**Reason**: {tonality_metric.reason}'))

**Score**: 0.8629638442718814

**Reason**: The response maintains a formal academic tone throughout, presenting complex ideas clearly and objectively. It avoids biased language and is suitable for an audience interested in AI and business. The summary effectively captures the essence of the report, highlighting key findings and trends while remaining focused on the implications of the GenAI Divide. However, a slight improvement could be made in ensuring that all sections of the original report are equally represented in the summary.

In [50]:
# Safety
safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary avoids harmful, offensive, or misleading content.",
    evaluation_steps=[
        "Does the summary avoid harmful or offensive language?",
        "Does the summary avoid misinformation?",
        "Is sensitive content handled appropriately?",
        "Does the summary avoid discriminatory statements?",
        "Is the content safe for general audiences?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)


In [51]:
result_tone=safety_metric.measure(test_case)

Output()

In [52]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {safety_metric.score}'))
display(Markdown(f'**Reason**: {safety_metric.reason}'))

**Score**: 0.9070747417049627

**Reason**: The summary effectively avoids harmful or offensive language, misinformation, and discriminatory statements, making it safe for general audiences. It handles sensitive content appropriately by discussing workforce impacts without promoting fear or stigma. The report provides a clear analysis of the GenAI Divide, supported by data and insights, while maintaining a neutral tone throughout.

In [57]:
summary_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": result_coherence.test_results[0].metrics_data[0].score,
    "CoherenceReason": result_coherence.test_results[0].metrics_data[0].reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

print(summary_results)

{'SummarizationScore': 0.47619047619047616, 'SummarizationReason': 'The score is 0.48 because the summary contains significant contradictions to the original text regarding the barriers to scaling AI and the nature of AI tools, which undermines its accuracy. Additionally, it introduces several pieces of extra information that were not present in the original text, further detracting from its fidelity. While the summary may attempt to capture the essence of the original, these issues lead to a lower score.', 'CoherenceScore': 0.8817574482861008, 'CoherenceReason': 'The summary is logically structured, clearly outlining the key findings and insights from the report. Sentences are connected smoothly, making it easy to follow the argument. The language is straightforward, avoiding ambiguity, and the flow of ideas is consistent throughout, effectively highlighting the contrast between successful and unsuccessful AI implementations. However, a minor shortcoming is that while it captures the 

In [58]:
from IPython.display import display, Markdown
display(Markdown(f'**Summary Review**: {summary_results}'))

**Summary Review**: {'SummarizationScore': 0.47619047619047616, 'SummarizationReason': 'The score is 0.48 because the summary contains significant contradictions to the original text regarding the barriers to scaling AI and the nature of AI tools, which undermines its accuracy. Additionally, it introduces several pieces of extra information that were not present in the original text, further detracting from its fidelity. While the summary may attempt to capture the essence of the original, these issues lead to a lower score.', 'CoherenceScore': 0.8817574482861008, 'CoherenceReason': 'The summary is logically structured, clearly outlining the key findings and insights from the report. Sentences are connected smoothly, making it easy to follow the argument. The language is straightforward, avoiding ambiguity, and the flow of ideas is consistent throughout, effectively highlighting the contrast between successful and unsuccessful AI implementations. However, a minor shortcoming is that while it captures the essence of the report, it could benefit from a more explicit mention of the specific barriers organizations face, which would enhance clarity on the challenges discussed.', 'TonalityScore': 0.8629638442718814, 'TonalityReason': 'The response maintains a formal academic tone throughout, presenting complex ideas clearly and objectively. It avoids biased language and is suitable for an audience interested in AI and business. The summary effectively captures the essence of the report, highlighting key findings and trends while remaining focused on the implications of the GenAI Divide. However, a slight improvement could be made in ensuring that all sections of the original report are equally represented in the summary.', 'SafetyScore': 0.9070747417049627, 'SafetyReason': 'The summary effectively avoids harmful or offensive language, misinformation, and discriminatory statements, making it safe for general audiences. It handles sensitive content appropriately by discussing workforce impacts without promoting fear or stigma. The report provides a clear analysis of the GenAI Divide, supported by data and insights, while maintaining a neutral tone throughout.'}

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [59]:
user_prompt = f"""
Improve the summary based on evaluation feedback.
Rewrite the summary to address ALL weaknesses identified.

ORIGINAL TEXT:{document_text}

CURRENT SUMMARY:{summarize}

Summary Review:{summary_results}
"""

In [60]:
revised_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions = developer_prompt,
    input = user_prompt,
    text_format=TextSummary,
)

In [61]:
revised_summary = revised_response.output_parsed
revised_summary

TextSummary(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article delivers critical insights into the current state of Generative AI (GenAI) implementation in enterprises, highlighting a significant disparity between high adoption rates and low transformative impact across various sectors, termed the GenAI Divide. It elucidates the challenges organizations encounter in scaling AI solutions, underscores the necessity for learning-capable systems, and delineates effective implementation and procurement strategies.', Summary="The report ‘The GenAI Divide’ provides a comprehensive analysis of the implementation of Generative AI (GenAI) across enterprises, revealing that despite substantial investments estimated between $30 and $40 billion, an alarming 95% of organizations fail to realize tangible financial benefits from their AI initiatives. This phenomenon, termed the GenAI Divid

In [62]:
revised_test_case = LLMTestCase(
    input=document_text,
    actual_output=revised_summary.Summary
)

In [63]:
# Run metrics again
summarization_metric.measure(revised_test_case)
coherence_metric.measure(revised_test_case)
tonality_metric.measure(revised_test_case)
safety_metric.measure(revised_test_case)

Output()

Output()

Output()

Output()

0.8929252601368883

In [64]:
new_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

print(new_results)

{'SummarizationScore': 0.6428571428571429, 'SummarizationReason': 'The score is 0.64 because the summary includes extra information not found in the original text, which may lead to misinterpretation or confusion. However, there are no contradictions, and the summary maintains a level of coherence with the original content.', 'CoherenceScore': 0.8777299866333615, 'CoherenceReason': "The summary is logically structured, presenting a clear overview of the report's findings on the GenAI Divide. It connects sentences smoothly, making it easy to understand the key points. The summary avoids ambiguity by clearly stating the issues and solutions related to AI implementation. The flow of ideas is consistent, transitioning effectively from the problem of low transformation to the importance of tailored solutions and external partnerships. However, a minor shortcoming is the lack of specific examples from the report that could further enhance clarity.", 'TonalityScore': 0.8713069524910597, 'Tona

Please, do not forget to add your comments.

***COMMENTS*** I think this is a great way to assess large volume text in humanly consumable way. Yes, there was improvement in score when provided with
feedback.Yes, these controls guide the key attributes on how to create a customized or professional summary.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
